<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase1/phase1_pix2struct_chartqa_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece evaluate
!pip install -q "pandas==2.2.2" "pillow<11.0"

In [ ]:
RUN_ID = datetime.now(ZoneInfo("America/Toronto")).strftime("%Y%m%d_%H%M%S")

CONFIG = {
    "phase": "phase1",
    "experiment_name": "pix2struct_chartqa_finetuned_baseline",

    "dataset_name": "HuggingFaceM4/ChartQA",
    "model_name": "google/pix2struct-chartqa-base",

    "split": "val",
    "max_samples": 50,
    "max_new_tokens": 64,

    "output_dir": "outputs/phase1",

    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "run_id": RUN_ID,
    "timezone": "America/Toronto"
}

CONFIG

In [ ]:
# git clone basic for future use
%cd /content

REPO_URL = "https://github.com/bogdanparvu18/msc-graduate-project.git"
REPO_NAME = "msc-graduate-project"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    %cd {REPO_NAME}
    !git pull
    %cd /content

%cd /content/msc-graduate-project
!pwd
!ls

In [ ]:
# ChartQA has already 3 splits, we select "val" first, then "test".

def load_dataset_from_config(config):
    """
    Loads the dataset specified in CONFIG.
    Side effect: downloads/loads dataset.
    """

    return load_dataset(config["dataset_name"])


def select_eval_data(dataset, config):
    """
    Selects the split and number of samples specified in CONFIG.
    """

    split_data = dataset[config["split"]]

    if config["max_samples"] is None:
        return split_data

    n = min(config["max_samples"], len(split_data))

    return split_data.select(range(n))


dataset = load_dataset_from_config(CONFIG)
eval_data = select_eval_data(dataset, CONFIG)

print(dataset)
print("Selected split:", CONFIG["split"])
print("Total examples in split:", len(dataset[CONFIG["split"]]))
print("Examples selected:", len(eval_data))

In [ ]:
sample = eval_data[0]

print("Question:", sample["query"])
print("Answer:", sample["label"])
print("Human(1) or machine(0) QA sample:", sample.get("human_or_machine", "N/A"))

sample["image"]

In [ ]:
def load_model_bundle(config):
    """
    Loads processor and model from CONFIG.
    Side effect: downloads/loads model into memory/GPU.
    """

    # Use AutoProcessor as it was imported in the setup cell
    processor = AutoProcessor.from_pretrained(config["model_name"])

    model = Pix2StructForConditionalGeneration.from_pretrained(
        config["model_name"]
    )

    model.to(config["device"])
    model.eval()

    return {
        "processor": processor,
        "model": model
    }


model_bundle = load_model_bundle(CONFIG)

print("Loaded model:", CONFIG["model_name"])
print("Device:", CONFIG["device"])

In [ ]:
def prepare_output_dirs(config):
    """
    Creates output directories for configs, results, and reports.
    Side effect: creates folders on temp disk.
    """

    output_dir = Path(config["output_dir"])

    dirs = {
        "output_dir": output_dir,
        "configs_dir": output_dir / "configs",
        "results_dir": output_dir / "results",
        "reports_dir": output_dir / "reports",
    }

    for directory in dirs.values():
        directory.mkdir(parents=True, exist_ok=True)

    return dirs


DIRS = prepare_output_dirs(CONFIG)

DIRS

In [ ]:
def normalize_answer(answer):
    """
    Converts an answer into a normalized string for exact-match evaluation.
    Pure-style function: same input -> same output.
    """

    if answer is None:
        return ""

    if isinstance(answer, list):
        answer = answer[0] if len(answer) > 0 else ""

    answer = str(answer).lower().strip()

    answer = re.sub(r"[,\.;:\!\?]", "", answer)
    answer = re.sub(r"\s+", " ", answer)

    return answer


def extract_number(text):
    """
    Extracts the first numeric value from text.
    Useful for answers such as '42', '42%', '$42.5', '1,200'.
    """

    if text is None:
        return None

    text = str(text).replace(",", "")
    match = re.search(r"-?\d+(\.\d+)?", text)

    return float(match.group()) if match else None


def numeric_match(gold, pred, tolerance=1e-3):
    """
    Compares the first numeric value from the gold answer and prediction.
    Returns 1 if they match within tolerance, otherwise 0.
    """

    gold_num = extract_number(gold)
    pred_num = extract_number(pred)

    if gold_num is None or pred_num is None:
        return 0

    return int(abs(gold_num - pred_num) <= tolerance)


def get_gold_answer(example):
    """
    Extracts the first gold answer from a ChartQA example.
    Handles both string and list labels.
    """

    label = example["label"]

    if isinstance(label, list):
        return label[0] if len(label) > 0 else ""

    return label

In [ ]:
def predict_one(example, model_bundle, config):
    """
    Generates one prediction for one ChartQA example.
    """

    processor = model_bundle["processor"]
    model = model_bundle["model"]

    image = example["image"]
    question = example["query"]

    if image.mode != "RGB":
        image = image.convert("RGB")

    inputs = processor(
        images=image,
        text=question,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(config["device"])
        for key, value in inputs.items()
    }

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=config["max_new_tokens"]
        )

    prediction = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return prediction.strip()

In [ ]:
sample = eval_data[0]

gold_answer = get_gold_answer(sample)
prediction = predict_one(sample, model_bundle, CONFIG)

print("Question:")
print(sample["query"])

print("\nGold answer:")
print(gold_answer)

print("\nPrediction:")
print(prediction)

print("\nNormalized gold:")
print(normalize_answer(gold_answer))

print("\nNormalized prediction:")
print(normalize_answer(prediction))

print("\nExact match:")
print(int(normalize_answer(gold_answer) == normalize_answer(prediction)))

print("\nNumeric match:")
print(numeric_match(gold_answer, prediction))

In [ ]:
def evaluate_one_example(example, index, model_bundle, config):
    """
    Runs prediction and evaluation for a single example.
    Returns one result row as a dictionary.
    """

    question = example["query"]
    gold_answer = get_gold_answer(example)
    gold_normalized = normalize_answer(gold_answer)

    base_row = {
        "index": index,
        "phase": config["phase"],
        "experiment_name": config["experiment_name"],
        "run_id": config["run_id"],
        "dataset_name": config["dataset_name"],
        "model_name": config["model_name"],
        "split": config["split"],
        "question": question,
        "gold_answer": gold_answer,
        "prediction": None,
        "gold_normalized": gold_normalized,
        "prediction_normalized": None,
        "exact_match": 0,
        "numeric_match": 0,
        "human_or_machine": example.get("human_or_machine", None),
        "error": None
    }

    try:
        prediction = predict_one(example, model_bundle, config)
        prediction_normalized = normalize_answer(prediction)

        return {
            **base_row,
            "prediction": prediction,
            "prediction_normalized": prediction_normalized,
            "exact_match": int(gold_normalized == prediction_normalized),
            "numeric_match": numeric_match(gold_answer, prediction)
        }

    except Exception as error:
        return {
            **base_row,
            "error": str(error)
        }

In [ ]:
def run_experiment(eval_data, model_bundle, config):
    """
    Runs inference and evaluation over all selected examples.
    Returns a DataFrame.
    """

    rows = [
        evaluate_one_example(example, index, model_bundle, config)
        for index, example in enumerate(
            tqdm(eval_data, desc="Running inference")
        )
    ]

    return pd.DataFrame(rows)


df_results = run_experiment(eval_data, model_bundle, CONFIG)

df_results.head()

In [ ]:
#compute metrics

def compute_metrics(df_results, config):
    """
    Computes aggregate evaluation metrics.
    """

    return {
        "phase": config["phase"],
        "experiment_name": config["experiment_name"],
        "run_id": config["run_id"],
        "dataset_name": config["dataset_name"],
        "model_name": config["model_name"],
        "split": config["split"],
        "max_samples": config["max_samples"],
        "max_new_tokens": config["max_new_tokens"],
        "seed": config["seed"],
        "device": config["device"],
        "num_examples_evaluated": int(len(df_results)),
        "num_errors": int(df_results["error"].notna().sum()),
        "exact_match_accuracy": float(df_results["exact_match"].mean()),
        "numeric_match_accuracy": float(df_results["numeric_match"].mean()),
        "timestamp": datetime.now(
            ZoneInfo(config["timezone"])
        ).isoformat()
    }


metrics = compute_metrics(df_results, CONFIG)

metrics

In [ ]:
def save_experiment_outputs(df_results, metrics, config, dirs):
    """
    Saves results CSV and metrics JSON.
    Side effect: writes files to disk.
    """

    base_name = f"{config['experiment_name']}_{config['run_id']}"

    results_path = dirs["results_dir"] / f"{base_name}_results.csv"
    metrics_path = dirs["results_dir"] / f"{base_name}_metrics.json"

    df_results.to_csv(results_path, index=False)

    save_json(metrics, metrics_path)

    return {
        "results_path": str(results_path),
        "metrics_path": str(metrics_path)
    }

saved_output_paths = save_experiment_outputs(
    df_results=df_results,
    metrics=metrics,
    config=CONFIG,
    dirs=DIRS
)

saved_output_paths

In [ ]:
wrong_answers = df_results[df_results["exact_match"] == 0]

print("Wrong answers:", len(wrong_answers))

wrong_answers[
    [
        "index",
        "question",
        "gold_answer",
        "prediction",
        "exact_match",
        "numeric_match",
        "error"
    ]
].head(20)

In [ ]:
# Set random seed for reproducibility
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
print(f"Seed set to: {CONFIG['seed']}")

In [ ]:

def make_json_serializable(obj):
    """
    Converts Python objects that are not directly JSON-serializable
    into JSON-compatible values.
    """

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, torch.device):
        return str(obj)

    if isinstance(obj, datetime):
        return obj.isoformat()

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    return str(obj)


def save_json(data, path):
    """
    Saves a dictionary as a JSON file.
    Side effect: writes file to disk.
    """

    with open(path, "w") as f:
        json.dump(
            data,
            f,
            indent=2,
            default=make_json_serializable
        )


def save_config_snapshot(config, dirs):
    """
    Saves the current CONFIG as a JSON snapshot.
    """

    config_path = (
        dirs["configs_dir"] /
        f"{config['experiment_name']}_{config['run_id']}_config.json"
    )

    save_json(config, config_path)

    return config_path


config_path = save_config_snapshot(CONFIG, DIRS)

print("Saved config snapshot to:")
print(config_path)

In [ ]:
def build_markdown_report(metrics, config, saved_paths):
    """
    Builds a Markdown report as a string.
    Pure-style function: returns text, does not write files.
    """

    report = f"""# Phase 1 - Generic VQA Baseline

## Experiment

- Phase: `{config["phase"]}`
- Experiment name: `{config["experiment_name"]}`
- Run ID: `{config["run_id"]}`
- Dataset: `{config["dataset_name"]}`
- Split: `{config["split"]}`
- Model: `{config["model_name"]}`
- Max samples: `{config["max_samples"]}`
- Max new tokens: `{config["max_new_tokens"]}`
- Seed: `{config["seed"]}`
- Device: `{config["device"]}`

## Objective

This experiment establishes an initial generic visual question answering baseline using a chart-based VQA dataset before moving to medical capsule endoscopy data.

## Method

The experiment uses a declarative configuration object and modular functional-style helper functions. The configuration defines the dataset, model, split, generation parameters, random seed, device, and output paths. The pipeline functions handle data loading, sample selection, inference, answer normalization, metric computation, and output serialization.

## Evaluation Metrics

Two simple metrics were used:

1. **Exact Match Accuracy**
   The generated answer and ground-truth answer are normalized and compared directly.

2. **Numeric Match Accuracy**
   The first numeric value is extracted from both the generated answer and the ground-truth answer and compared using a small tolerance.

## Results

- Number of evaluated examples: `{metrics["num_examples_evaluated"]}`
- Errors during inference: `{metrics["num_errors"]}`
- Exact Match Accuracy: `{metrics["exact_match_accuracy"]:.4f}`
- Numeric Match Accuracy: `{metrics["numeric_match_accuracy"]:.4f}`

## Output Files

- Results CSV: `{saved_paths["results_path"]}`
- Metrics JSON: `{saved_paths["metrics_path"]}`

## Notes

The model `google/pix2struct-chartqa-base` is already fine-tuned on ChartQA. Therefore, this experiment should be treated as a fine-tuned baseline rather than a strict zero-shot experiment.

The next step is to repeat the same pipeline with additional datasets and models, such as DVQA, DashboardQA, DePlot, Qwen2.5-VL, InternVL, and LLaVA-style models.
"""

    return report


def save_report(report_text, config, dirs):
    """
    Saves a Markdown report.
    Side effect: writes file to disk.
    """

    base_name = f"{config['experiment_name']}_{config['run_id']}"
    report_path = dirs["reports_dir"] / f"{base_name}_report.md"

    with open(report_path, "w") as f:
        f.write(report_text)

    return report_path


report_text = build_markdown_report(metrics, CONFIG, saved_output_paths)
report_path = save_report(report_text, CONFIG, DIRS)

print("Saved report to:")
print(report_path)

In [ ]:
print("Experiment completed.")
print()
print("Experiment name:", CONFIG["experiment_name"])
print("Run ID:", CONFIG["run_id"])
print("Dataset:", CONFIG["dataset_name"])
print("Model:", CONFIG["model_name"])
print("Split:", CONFIG["split"])
print("Samples:", metrics["num_examples_evaluated"])
print("Exact Match Accuracy:", metrics["exact_match_accuracy"])
print("Numeric Match Accuracy:", metrics["numeric_match_accuracy"])
print()
print("Config:")
print(config_path)
print()
print("Results:")
print(saved_output_paths["results_path"])
print()
print("Metrics:")
print(saved_output_paths["metrics_path"])
print()
print("Report:")
print(report_path)

In [ ]:
!git status

In [ ]:
!git add outputs/phase1/

In [ ]:
!git remote -v

In [ ]:
!git push origin main

In [ ]:
!git config --global user.name "Bogdan Parvu"
!git add outputs/phase1/
!git commit -m "Add Phase 1 Pix2Struct ChartQA baseline run"

In [ ]:
# Phase 1a — Generic VQA Baseline with Pix2Struct on ChartQA
# Goal:
#   Load a non-medical visual-question-answering dataset.
#   Run a Pix2Struct baseline.
#   Generate answers for a small evaluation subset.
#   Save predictions in JSONL format.
#   Compute simple exact-match / relaxed-match accuracy.
#   This is the start of the Phase 1 work
import os
import re
import json
import random
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import torch
import pandas as pd

from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoProcessor, Pix2StructForConditionalGeneration